In [2]:
import json
from pathlib import Path 

wandb_dir = Path("wandb_final_scorers")
exp_name = "scorers-ho-heavy-backbone"
# exp_name = "scorers-ho-v5"
input_file = wandb_dir / exp_name / "results.json"

with input_file.open("r") as f:
    data = json.load(f)

In [3]:
data.pop("null")

KeyError: 'null'

In [4]:
list(data.keys())

["banking77[seed='42']",
 "minds14[seed='42']",
 "hwu64[seed='42']",
 "massive[seed='42']",
 "banking77[seed='43']",
 "minds14[seed='43']",
 "hwu64[seed='43']",
 "massive[seed='43']",
 "banking77[seed='44']",
 "minds14[seed='44']",
 "hwu64[seed='44']",
 "massive[seed='44']"]

In [5]:
def extract_dataset_and_seed(key):
    """
    Extract dataset name and seed from a string like "banking77[seed='44']"
    
    Args:
        key (str): String in format "dataset_name[seed='seed_value']"
        
    Returns:
        tuple: (dataset_name, seed_value)
    """
    import re
    
    # Use regex to extract dataset name and seed value
    match = re.match(r"([^[]+)\[seed='(\d+)'\]", key)
    
    if match:
        dataset_name = match.group(1)
        seed = match.group(2)
        return dataset_name, seed
    else:
        return None, None


In [6]:
from collections import defaultdict

dataset_wise_results = defaultdict(list)
for key in data.keys():
    dataset, seed = extract_dataset_and_seed(key)
    dataset_wise_results[dataset].append(data[key])

In [7]:
len(dataset_wise_results["banking77"][0])

1

In [8]:
dataset_wise_results

defaultdict(list,
            {'banking77': [[{'module_name': 'final_metrics',
                'config': {'value': {'decision_accuracy': 0.9256493506493506,
                  'decision_f1': 0.9258010408032752,
                  'decision_precision': 0.9287298315949438,
                  'decision_recall': 0.9256493506493509,
                  'decision_roc_auc': 0.9623355263157897}}}],
              [{'module_name': 'final_metrics',
                'config': {'value': {'decision_accuracy': 0.9298701298701298,
                  'decision_f1': 0.9298708205089007,
                  'decision_precision': 0.9338653933914957,
                  'decision_recall': 0.9298701298701298,
                  'decision_roc_auc': 0.9644736842105264}}}],
              [{'module_name': 'final_metrics',
                'config': {'value': {'decision_accuracy': 0.9301948051948052,
                  'decision_f1': 0.9301431511050137,
                  'decision_precision': 0.9339972057445086,
              

- best result for each scorer
- 

In [7]:
import pandas as pd


dataset_wise_aggregated = {}
for dataset_name, dataset_results in dataset_wise_results.items():
    aggregated = []
    for seed_results in dataset_results:
        scorer_wise_results = defaultdict(list)
        for run in seed_results:
            try:
                module_name = run["module_name"].split("_")[0]
                metrics = run.get("wandb-summary", {})
                scorer_wise_results[module_name].append(metrics)
            except Exception:
                print("error processing run:", run)
                raise
        for scorer_name, scorer_runs in scorer_wise_results.items():
            best_run = max(scorer_runs, key=lambda x: x.get("scoring_accuracy", 0))
            if "scoring_accuracy" not in best_run:
                continue
            metrics = {
                "scoring_accuracy": 100 * best_run["scoring_accuracy"],
                "scoring_recall": 100 * best_run["scoring_recall"],
                "scoring_precision": 100 * best_run["scoring_precision"],
                "scoring_roc_auc": 100 * best_run["scoring_roc_auc"],
                "scoring_f1": 100 * best_run["scoring_f1"],
            }
            aggregated.append({"module_name": scorer_name, **metrics})
    dataset_wise_aggregated[dataset_name] = pd.DataFrame(aggregated)


In [8]:
res = None
for dataset_name, dataset_results in dataset_wise_aggregated.items():
    averaged = dataset_results.groupby("module_name").mean().sort_values(by="scoring_accuracy")
    averaged.rename(columns={"scoring_accuracy": dataset_name}, inplace=True)
    averaged.drop(columns=["scoring_f1", "scoring_roc_auc", "scoring_recall", "scoring_precision"], inplace=True)
    if res is None:
        res = averaged
    else:
        res = pd.merge(left=res, right=averaged, on="module_name")
res = res.reset_index()

In [9]:
res

,module_name,banking77,minds14,hwu64,snips,massive
0,ptuning,4.628705,11.627907,3.645833,66.488414,8.567208
1,lora,20.346320,12.403101,22.395833,95.085307,41.949778
2,bert,64.135864,69.767442,73.400298,98.523046,76.760217
3,dnnc,88.877789,97.674419,82.514881,95.416348,78.631216
4,rerank,89.044289,97.674419,84.449405,96.587726,81.683900
5,knn,89.743590,97.674419,85.416667,96.103896,81.651075
6,sklearn,89.810190,98.449612,86.979167,95.161701,79.730839
7,linear,90.509491,97.674419,89.174107,97.453527,84.703758


In [10]:
res.to_string()

'  module_name  banking77    minds14      hwu64      snips    massive\n0     ptuning   4.628705  11.627907   3.645833  66.488414   8.567208\n1        lora  20.346320  12.403101  22.395833  95.085307  41.949778\n2        bert  64.135864  69.767442  73.400298  98.523046  76.760217\n3        dnnc  88.877789  97.674419  82.514881  95.416348  78.631216\n4      rerank  89.044289  97.674419  84.449405  96.587726  81.683900\n5         knn  89.743590  97.674419  85.416667  96.103896  81.651075\n6     sklearn  89.810190  98.449612  86.979167  95.161701  79.730839\n7      linear  90.509491  97.674419  89.174107  97.453527  84.703758'

In [11]:
res["average"] = res.drop(columns=["module_name"]).mean(axis=1)
res.sort_values(by="average")

,module_name,banking77,minds14,hwu64,snips,massive,average
0,ptuning,4.628705,11.627907,3.645833,66.488414,8.567208,18.991613
1,lora,20.346320,12.403101,22.395833,95.085307,41.949778,38.436068
2,bert,64.135864,69.767442,73.400298,98.523046,76.760217,76.517373
3,dnnc,88.877789,97.674419,82.514881,95.416348,78.631216,88.622931
4,rerank,89.044289,97.674419,84.449405,96.587726,81.683900,89.887948
6,sklearn,89.810190,98.449612,86.979167,95.161701,79.730839,90.026302
5,knn,89.743590,97.674419,85.416667,96.103896,81.651075,90.117929
7,linear,90.509491,97.674419,89.174107,97.453527,84.703758,91.903060


In [12]:
def count_winnings(df):
    """
    Adds a column to the DataFrame that counts how many times each row contains
    a maximum value in any column. Handles ties by counting all max values.
    
    Parameters:
    df (pd.DataFrame): Input DataFrame with results for multiple datasets
    
    Returns:
    pd.DataFrame: DataFrame with an additional 'best_count' column
    """
    
    # Initialize the count column with zeros
    df['best_count'] = 0
    
    # Iterate through each column
    for col in df.columns:
        if col in ['best_count', 'average']:
            continue  # Skip our new count column
            
        # Find the max value in the column
        max_val = df[col].max()
        
        # Increment count for all rows that have the max value in this column
        df['best_count'] += (df[col] == max_val).astype(int)
    

In [13]:
count_winnings(res)

In [14]:
res.style.highlight_max(color='gray')

,module_name,banking77,minds14,hwu64,snips,massive,average,best_count
0,ptuning,4.628705,11.627907,3.645833,66.488414,8.567208,18.991613,0
1,lora,20.346320,12.403101,22.395833,95.085307,41.949778,38.436068,0
2,bert,64.135864,69.767442,73.400298,98.523046,76.760217,76.517373,1
3,dnnc,88.877789,97.674419,82.514881,95.416348,78.631216,88.622931,0
4,rerank,89.044289,97.674419,84.449405,96.587726,81.683900,89.887948,0
5,knn,89.743590,97.674419,85.416667,96.103896,81.651075,90.117929,0
6,sklearn,89.810190,98.449612,86.979167,95.161701,79.730839,90.026302,2
7,linear,90.509491,97.674419,89.174107,97.453527,84.703758,91.903060,3


In [15]:
print(res.sort_values(by="average").round(2).to_string())

  module_name  banking77  minds14  hwu64  snips  massive  average  best_count
0     ptuning       4.63    11.63   3.65  66.49     8.57    18.99           0
1        lora      20.35    12.40  22.40  95.09    41.95    38.44           0
2        bert      64.14    69.77  73.40  98.52    76.76    76.52           1
3        dnnc      88.88    97.67  82.51  95.42    78.63    88.62           0
4      rerank      89.04    97.67  84.45  96.59    81.68    89.89           0
6     sklearn      89.81    98.45  86.98  95.16    79.73    90.03           2
5         knn      89.74    97.67  85.42  96.10    81.65    90.12           0
7      linear      90.51    97.67  89.17  97.45    84.70    91.90           3
